# Batch Normalization（批归一化）

Batch Normalization（BN）在每个通道上标准化激活值，再通过可学习的缩放和偏移恢复表达能力。它有助于稳定训练过程，并允许模型使用更合适的学习率。

对输入 $X\in\mathbb{R}^{N\times C\times H\times W}$，训练时按通道在 $N,H,W$ 维度计算：

$$\mu_c=\frac{1}{M}\sum_{i=1}^{M}x_{i,c},\qquad \sigma_c^2=\frac{1}{M}\sum_{i=1}^{M}(x_{i,c}-\mu_c)^2,\quad M=N\cdot H\cdot W.$$

归一化与仿射变换为：

$$\hat{x}_{i,c}=\frac{x_{i,c}-\mu_c}{\sqrt{\sigma_c^2+\varepsilon}},\qquad y_{i,c}=\gamma_c\hat{x}_{i,c}+\beta_c.$$

训练期间还以动量 $\alpha$ 更新运行统计量 $r\leftarrow(1-\alpha)r+\alpha r_{batch}$；推理时使用这些运行均值和方差。

In [ ]:
import torch
import torch.nn as nn

class MyBatchNorm2d(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super(MyBatchNorm2d, self).__init__()
        self.num_features = num_features
        self.eps = eps
        self.momentum = momentum
        
        # 可学习的每通道仿射参数，形状都是 [C]；forward 中会扩展为 [1, C, 1, 1]。
        self.weight = nn.Parameter(torch.ones(num_features))  # gamma: [C]
        self.bias = nn.Parameter(torch.zeros(num_features))   # beta:  [C]
        
        # 运行统计量不是参数，但需随模型保存、加载与迁移设备；形状均为 [C]。
        self.register_buffer('running_mean', torch.zeros(num_features))  # [C]
        self.register_buffer('running_var', torch.ones(num_features))    # [C]
        
    def forward(self, x):
        # x: [N, C, H, W]；所有统计量均按通道 C 独立计算。
        if self.training:
            # 1. 在 N、H、W 上归约，保留 C： [N, C, H, W] -> [C]。
            mean = x.mean(dim=(0, 2, 3))  # [C]
            var = x.var(dim=(0, 2, 3), unbiased=False)  # [C]，有偏方差
            
            # 2. [C] 与 [C] 逐元素更新；no_grad 防止统计量进入反向传播图。
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * var
        else:
            # 推理模式：直接取累计的 [C] 运行均值与方差。
            mean = self.running_mean
            var = self.running_var
            
        # 3. [C] -> [1, C, 1, 1]，从而可广播到输入的 [N, C, H, W]。
        x_hat = (x - mean.view(1, -1, 1, 1)) / torch.sqrt(var.view(1, -1, 1, 1) + self.eps)  # [N, C, H, W]
        
        # 4. gamma、beta 同样扩展为 [1, C, 1, 1]；输出形状不变。
        out = x_hat * self.weight.view(1, -1, 1, 1) + self.bias.view(1, -1, 1, 1)  # [N, C, H, W]
        return out

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_head, dropout=0.0):
        super().__init__()
        assert d_model % num_head == 0, "d_model must be divisible by num_head"
        
        self.d_model = d_model
        self.num_head = num_head
        self.d_k = d_model // num_head
        
        # 最后一维 D 映射为 3D，供后续拆分 Query、Key、Value。
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        # x: [B, L, D]，分别为批大小、序列长度和 d_model。
        B, L, D = x.shape
        
        # 1. Linear: [B, L, D] -> [B, L, 3D]；reshape 后显式分出 Q/K/V 与头维。
        # qkv: [B, L, 3, H, d_k] -> permute -> [3, B, H, L, d_k]
        qkv = self.qkv(x).reshape(B, L, 3, self.num_head, self.d_k).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # 每个都是 [B, H, L, d_k]
        
        # 2. 计算缩放点积注意力 (推荐用 matmul 替代 einsum)
        # q: [B, H, L, d_k] @ k^T: [B, H, d_k, L] -> attn: [B, H, L, L]
        attn_score = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        # attn_score=torch.einsum("b h i d,b h j d->b h i j",q,k)/(self.d_k**0.5)
        
        # 3. 处理 Mask (面试加分项：兼容多种 mask 形状)
        if mask is not None:
            # mask 为 [B, L] 时扩展为 [B, 1, 1, L]，广播到每个头和每个 query 位置。
            # mask 为 [L, L] 时同样扩展，可广播到批次与头维。
            if mask.dim() == 2:
                mask = mask.unsqueeze(1).unsqueeze(2)  # [B, 1, 1, L] 或 [1, 1, L, L]
            elif mask.dim() == 3:
                mask = mask.unsqueeze(1)  # [B, 1, L, L]
            
            attn_score = attn_score.masked_fill(mask == 0, -float('inf'))
        
        # 4. 在最后一个 L（key）维归一化，注意力权重形状仍为 [B, H, L, L]。
        attn = F.softmax(attn_score, dim=-1)
        attn = self.dropout(attn)  # 形状保持 [B, H, L, L]
        
        # 5. 加权求和并合并多头
        # attn: [B, H, L, L] @ v: [B, H, L, d_k] -> out: [B, H, L, d_k]
        out = torch.matmul(attn, v)
        # out=torch.einsum("b h i j, b h j d->b h i d",attn,v)
        # 合并多头: [B, H, L, d_k] -> [B, L, H, d_k] -> [B, L, D]
        out = out.transpose(1, 2).reshape(B, L, D)
        
        # 6. 输出投影只作用于最后一维 D，形状保持 [B, L, D]。
        return self.out_proj(out)